In [5]:
import pandas as pd
from sklearn import svm
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from imblearn.over_sampling import SMOTE
from collections import defaultdict

# Before Feature Selection

In [6]:
file_path = "Dataset/kids_price.csv"

# Read the first line to count the number of columns
with open(file_path, 'r', encoding='utf-8') as x:
    ncols = len(x.readline().strip().split(','))

# Load CSV using the correct number of columns
df_kids = pd.read_csv(file_path, usecols=range(0, ncols))

# Display the DataFrame
df_kids.head()

,UMUR (BULAN),BANGSA,AGAMA,JANTINA,PENDAPATAN KELUARGA,GAJI BAPA,GAJI IBU,GAJI PENJAGA,STATUS PEMAKANAN,DAERAH,...,banana_price,papaya_price,rice_price,bread_price,fish_price,chicken_price,carrot_price,tomato_price,cauliflower_price,milk_price
0,60,Melayu,Islam,LELAKI,M40,"RM 4,000 - RM 6,999","RM 4,000 - RM 6,999","RM 4,000 - RM 6,999",Malpemakanan,Seberang Perai Tengah,...,6.0,4.29,25.99,2.8,9.99,8.9,4.49,5.00,7.9,20.9
1,54,Melayu,Islam,LELAKI,Tiada Maklumat,"RM 10,000 dan ke atas","RM 4,000 - RM 6,999","RM 4,000 - RM 6,999",Malpemakanan,Seberang Perai Tengah,...,6.0,4.29,25.99,2.8,9.99,8.9,4.49,5.00,7.9,20.9
2,53,Melayu,Islam,PEREMPUAN,B40,"RM 1,000 - RM 3,999","RM 1,000 - RM 3,999","RM 1,000 - RM 3,999",Malpemakanan,Seberang Perai Tengah,...,6.0,4.29,25.99,2.8,9.99,8.9,4.49,5.00,7.9,20.9
3,52,Melayu,Islam,PEREMPUAN,B40,"RM 1,000 - RM 3,999","RM 1,000 - RM 3,999","RM 1,000 - RM 3,999",Malpemakanan,Seberang Perai Tengah,...,6.0,4.29,25.99,2.8,9.99,8.9,4.49,5.00,7.9,20.9
4,58,Melayu,Islam,LELAKI,B40,"RM 1,000 - RM 3,999",TIADA MAKLUMAT GAJI,"RM 1,000 - RM 3,999",Malpemakanan,Daerah Barat Daya,...,6.0,4.50,25.90,2.8,10.00,8.9,5.00,5.49,8.0,20.9


## Encoding Strategy

### Label Encoder

In [7]:
# Create a LabelEncoder object
le = LabelEncoder()
df_kids_le = df_kids.copy()

# Iterate over the columns of the DataFrame
for col in df_kids_le.columns:
    # Check if the column is of object type (categorical)
    if df_kids_le[col].dtype == 'object':
        # Fit and transform the column using LabelEncoder
        df_kids_le[col] = le.fit_transform(df_kids_le[col])

df_kids_le.head()

,UMUR (BULAN),BANGSA,AGAMA,JANTINA,PENDAPATAN KELUARGA,GAJI BAPA,GAJI IBU,GAJI PENJAGA,STATUS PEMAKANAN,DAERAH,...,banana_price,papaya_price,rice_price,bread_price,fish_price,chicken_price,carrot_price,tomato_price,cauliflower_price,milk_price
0,60,5,2,0,1,4,4,4,1,3,...,6.0,4.29,25.99,2.8,9.99,8.9,4.49,5.00,7.9,20.9
1,54,5,2,0,4,3,4,4,1,3,...,6.0,4.29,25.99,2.8,9.99,8.9,4.49,5.00,7.9,20.9
2,53,5,2,1,0,2,2,2,1,3,...,6.0,4.29,25.99,2.8,9.99,8.9,4.49,5.00,7.9,20.9
3,52,5,2,1,0,2,2,2,1,3,...,6.0,4.29,25.99,2.8,9.99,8.9,4.49,5.00,7.9,20.9
4,58,5,2,0,0,2,6,2,1,0,...,6.0,4.50,25.90,2.8,10.00,8.9,5.00,5.49,8.0,20.9


In [8]:
x_le = df_kids_le.drop('BMI', axis=1)
y_le = df_kids_le['BMI']

x_train_le, x_test_le, y_train_le, y_test_le = train_test_split( x_le, y_le, test_size = 0.30, random_state=42, stratify=y_le)

In [9]:
svm_model = svm.SVC(kernel='rbf', class_weight='balanced', random_state=42)
svm_model.fit(x_train_le, y_train_le)
y_pred = svm_model.predict(x_test_le)

In [10]:
# Evaluate the model
print("Confusion Matrix:")
print(confusion_matrix(y_test_le, y_pred))
print("\nClassification Report:")
print(classification_report(y_test_le, y_pred))

Confusion Matrix:
[[44  0  0  0 19  2]
 [22  0  1  0  7  1]
 [16  0  0  0 10  1]
 [35  0  0  0 12  0]
 [14  0  0  0  9  1]
 [ 6  0  0  0  5  1]]

Classification Report:
              precision    recall  f1-score   support

           0       0.32      0.68      0.44        65
           1       0.00      0.00      0.00        31
           2       0.00      0.00      0.00        27
           3       0.00      0.00      0.00        47
           4       0.15      0.38      0.21        24
           5       0.17      0.08      0.11        12

    accuracy                           0.26       206
   macro avg       0.11      0.19      0.13       206
weighted avg       0.13      0.26      0.17       206



c:\Users\hp\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\hp\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\hp\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f

### One-Hot Encoder

In [11]:
df_kids_ohe = df_kids.copy()
df_kids_ohe = df_kids_ohe.drop('BMI', axis=1)

In [12]:
# Create a OneHotEncoder object
ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=False)

# Fit and transform the categorical columns
categorical_cols = df_kids_ohe.select_dtypes(include=['object']).columns
df_kids_encoded = pd.DataFrame(ohe.fit_transform(df_kids_ohe[categorical_cols]))

# Get feature names for the encoded columns
encoded_feature_names = list(ohe.get_feature_names_out(categorical_cols))
df_kids_encoded.columns = encoded_feature_names

# Drop original categorical columns from the dataframe
df_kids_ohe = df_kids_ohe.drop(categorical_cols, axis=1)

# Concatenate the encoded columns with the remaining numerical features
df_kids_ohe = pd.concat([df_kids_ohe, df_kids_encoded], axis=1)

df_kids_ohe.head()

,UMUR (BULAN),banana_price,papaya_price,rice_price,bread_price,fish_price,chicken_price,carrot_price,tomato_price,cauliflower_price,...,DAERAH_Seberang Perai Selatan,DAERAH_Seberang Perai Tengah,DAERAH_Seberang Perai Utara,"JENIS TASKA_TASKA Agensi Kerajaan (GENIUS, KEMAS, JPNIN, YPKT)",JENIS TASKA_TASKA Di Rumah,JENIS TASKA_TASKA Di Tempat Kerja (Sektor Awam),JENIS TASKA_TASKA Di Tempat Kerja (Sektor Swasta),JENIS TASKA_TASKA Institusi,TASKA_LOKASI_BANDAR,TASKA_LOKASI_LUAR BANDAR
0,60,6.0,4.29,25.99,2.8,9.99,8.9,4.49,5.00,7.9,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0
1,54,6.0,4.29,25.99,2.8,9.99,8.9,4.49,5.00,7.9,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0
2,53,6.0,4.29,25.99,2.8,9.99,8.9,4.49,5.00,7.9,...,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0
3,52,6.0,4.29,25.99,2.8,9.99,8.9,4.49,5.00,7.9,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0
4,58,6.0,4.50,25.90,2.8,10.00,8.9,5.00,5.49,8.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0


In [13]:
x_ohe = df_kids_ohe
y_ohe = df_kids['BMI']

x_train_ohe, x_test_ohe, y_train_ohe, y_test_ohe = train_test_split( x_ohe, y_ohe, test_size = 0.30, random_state=42, stratify=y_ohe)

In [14]:
svm_model = svm.SVC(kernel='rbf', class_weight='balanced', random_state=42)
svm_model.fit(x_train_ohe, y_train_ohe)
y_pred = svm_model.predict(x_test_ohe)

In [15]:
# Evaluate the model
print("Confusion Matrix:")
print(confusion_matrix(y_test_ohe, y_pred))
print("\nClassification Report:")
print(classification_report(y_test_ohe, y_pred))

Confusion Matrix:
[[37  0  0  0 26  2]
 [20  0  0  0 11  0]
 [14  0  0  0 13  0]
 [30  0  0  0 17  0]
 [13  0  0  0 11  0]
 [ 6  0  0  0  6  0]]

Classification Report:
                               precision    recall  f1-score   support

           Berat badan normal       0.31      0.57      0.40        65
       Berlebihan berat badan       0.00      0.00      0.00        31
                         Obes       0.00      0.00      0.00        27
Risiko berlebihan berat badan       0.00      0.00      0.00        47
                        Susut       0.13      0.46      0.20        24
                  Susut teruk       0.00      0.00      0.00        12

                     accuracy                           0.23       206
                    macro avg       0.07      0.17      0.10       206
                 weighted avg       0.11      0.23      0.15       206



c:\Users\hp\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\hp\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\hp\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f

## Use SMOTE(Synthetic Minority Over-sampling Technique) to solve class imbalance

### Label Encoder

In [16]:
sm = SMOTE(random_state=42)
X_train_res, y_train_res = sm.fit_resample(x_train_le, y_train_le)

In [17]:
svm_model = svm.SVC(kernel='rbf', class_weight='balanced', random_state=42)
svm_model.fit(X_train_res, y_train_res)
y_pred = svm_model.predict(x_test_le)

In [18]:
# Evaluate the model
print("Confusion Matrix:")
print(confusion_matrix(y_test_le, y_pred))
print("\nClassification Report:")
print(classification_report(y_test_le, y_pred))

Confusion Matrix:
[[ 4  3 15 13  2 28]
 [ 0  0  7  7  1 16]
 [ 0  2 10  3  0 12]
 [ 5  0 11  8  0 23]
 [ 2  1  9  6  0  6]
 [ 2  0  4  1  1  4]]

Classification Report:
              precision    recall  f1-score   support

           0       0.31      0.06      0.10        65
           1       0.00      0.00      0.00        31
           2       0.18      0.37      0.24        27
           3       0.21      0.17      0.19        47
           4       0.00      0.00      0.00        24
           5       0.04      0.33      0.08        12

    accuracy                           0.13       206
   macro avg       0.12      0.16      0.10       206
weighted avg       0.17      0.13      0.11       206



### One-Hot Encoder

In [19]:
sm = SMOTE(random_state=42)
X_train_res, y_train_res = sm.fit_resample(x_train_ohe, y_train_ohe)

In [20]:
svm_model = svm.SVC(kernel='rbf', class_weight='balanced', random_state=42)
svm_model.fit(X_train_res, y_train_res)
y_pred = svm_model.predict(x_test_ohe)

In [21]:
# Evaluate the model
print("Confusion Matrix:")
print(confusion_matrix(y_test_ohe, y_pred))
print("\nClassification Report:")
print(classification_report(y_test_ohe, y_pred))

Confusion Matrix:
[[ 0 12 20  9  0 24]
 [ 0  5  8  4  0 14]
 [ 0  4 10  7  0  6]
 [ 0  6 14  8  0 19]
 [ 0  4 10  5  0  5]
 [ 0  2  5  2  0  3]]

Classification Report:
                               precision    recall  f1-score   support

           Berat badan normal       0.00      0.00      0.00        65
       Berlebihan berat badan       0.15      0.16      0.16        31
                         Obes       0.15      0.37      0.21        27
Risiko berlebihan berat badan       0.23      0.17      0.20        47
                        Susut       0.00      0.00      0.00        24
                  Susut teruk       0.04      0.25      0.07        12

                     accuracy                           0.13       206
                    macro avg       0.10      0.16      0.11       206
                 weighted avg       0.10      0.13      0.10       206



c:\Users\hp\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\hp\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\hp\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f

# After Feature Selection

## Strategy: Drop Invalid Data

In [22]:
file_path = "Dataset/DROP_INVALID_AfterFS.csv"

# Read the first line to count the number of columns
with open(file_path, 'r', encoding='utf-8') as x:
    ncols = len(x.readline().strip().split(','))

# Load CSV using the correct number of columns
df_DROP_INVALID_AfterFS = pd.read_csv(file_path, usecols=range(0, ncols))

# Display the DataFrame
df_DROP_INVALID_AfterFS.head()

,UMUR (BULAN),STATUS PEMAKANAN,JENIS TASKA,TASKA_LOKASI,BMI,banana_price,papaya_price,rice_price,bread_price,fish_price,chicken_price,carrot_price,tomato_price,cauliflower_price,milk_price
0,60,Malpemakanan,TASKA Institusi,BANDAR,Risiko berlebihan berat badan,6.0,4.29,25.99,2.8,9.99,8.9,4.49,5.00,7.9,20.9
1,53,Malpemakanan,TASKA Di Rumah,BANDAR,Risiko berlebihan berat badan,6.0,4.29,25.99,2.8,9.99,8.9,4.49,5.00,7.9,20.9
2,52,Malpemakanan,TASKA Institusi,BANDAR,Obes,6.0,4.29,25.99,2.8,9.99,8.9,4.49,5.00,7.9,20.9
3,58,Malpemakanan,"TASKA Agensi Kerajaan (GENIUS, KEMAS, JPNIN, Y...",LUAR BANDAR,Berlebihan berat badan,6.0,4.50,25.90,2.8,10.00,8.9,5.00,5.49,8.0,20.9
4,56,Malpemakanan,"TASKA Agensi Kerajaan (GENIUS, KEMAS, JPNIN, Y...",LUAR BANDAR,Berat badan normal,6.0,4.50,25.90,2.8,10.00,8.9,5.00,5.49,8.0,20.9


### Encoding Strategy

#### Label Encoder

In [23]:
# Create a LabelEncoder object
le = LabelEncoder()
df_DROP_INVALID_AfterFS_le = df_DROP_INVALID_AfterFS.copy()

# Iterate over the columns of the DataFrame
for col in df_DROP_INVALID_AfterFS_le.columns:
    # Check if the column is of object type (categorical)
    if df_DROP_INVALID_AfterFS_le[col].dtype == 'object':
        # Fit and transform the column using LabelEncoder
        df_DROP_INVALID_AfterFS_le[col] = le.fit_transform(df_DROP_INVALID_AfterFS_le[col])

df_DROP_INVALID_AfterFS_le.head()

,UMUR (BULAN),STATUS PEMAKANAN,JENIS TASKA,TASKA_LOKASI,BMI,banana_price,papaya_price,rice_price,bread_price,fish_price,chicken_price,carrot_price,tomato_price,cauliflower_price,milk_price
0,60,0,4,0,3,6.0,4.29,25.99,2.8,9.99,8.9,4.49,5.00,7.9,20.9
1,53,0,1,0,3,6.0,4.29,25.99,2.8,9.99,8.9,4.49,5.00,7.9,20.9
2,52,0,4,0,2,6.0,4.29,25.99,2.8,9.99,8.9,4.49,5.00,7.9,20.9
3,58,0,0,1,1,6.0,4.50,25.90,2.8,10.00,8.9,5.00,5.49,8.0,20.9
4,56,0,0,1,0,6.0,4.50,25.90,2.8,10.00,8.9,5.00,5.49,8.0,20.9


In [24]:
x_le = df_DROP_INVALID_AfterFS_le.drop('BMI', axis=1)
y_le = df_DROP_INVALID_AfterFS_le['BMI']

x_train_le, x_test_le, y_train_le, y_test_le = train_test_split(x_le, y_le, test_size = 0.30, random_state=42, stratify=y_le)

In [25]:
svm_model = svm.SVC(kernel='rbf', class_weight='balanced', random_state=42)
svm_model.fit(x_train_le, y_train_le)
y_pred = svm_model.predict(x_test_le)

In [26]:
# Evaluate the model
print("Confusion Matrix:")
print(confusion_matrix(y_test_le, y_pred))
print("\nClassification Report:")
print(classification_report(y_test_le, y_pred))

Confusion Matrix:
[[ 0 12  0 22 21  0]
 [ 0  4  0 13  7  0]
 [ 0  3  0  6  9  2]
 [ 0 12  0 18 10  0]
 [ 0  5  0  6  7  0]
 [ 0  2  0  4  2  2]]

Classification Report:
              precision    recall  f1-score   support

           0       0.00      0.00      0.00        55
           1       0.11      0.17      0.13        24
           2       0.00      0.00      0.00        20
           3       0.26      0.45      0.33        40
           4       0.12      0.39      0.19        18
           5       0.50      0.20      0.29        10

    accuracy                           0.19       167
   macro avg       0.17      0.20      0.16       167
weighted avg       0.12      0.19      0.14       167



c:\Users\hp\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\hp\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\hp\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f

#### One-Hot Encoder

In [27]:
df_DROP_INVALID_AfterFS_ohe = df_DROP_INVALID_AfterFS.copy()
df_DROP_INVALID_AfterFS_ohe = df_DROP_INVALID_AfterFS_ohe.drop('BMI', axis=1)

In [28]:
# Create a OneHotEncoder object
ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=False)

# Fit and transform the categorical columns
categorical_cols = df_DROP_INVALID_AfterFS_ohe.select_dtypes(include=['object']).columns
df_DROP_INVALID_AfterFS_encoded = pd.DataFrame(ohe.fit_transform(df_DROP_INVALID_AfterFS_ohe[categorical_cols]))

# Get feature names for the encoded columns
encoded_feature_names = list(ohe.get_feature_names_out(categorical_cols))
df_DROP_INVALID_AfterFS_encoded.columns = encoded_feature_names

# Drop original categorical columns from the dataframe
df_DROP_INVALID_AfterFS_ohe = df_DROP_INVALID_AfterFS_ohe.drop(categorical_cols, axis=1)

# Concatenate the encoded columns with the remaining numerical features
df_DROP_INVALID_AfterFS_ohe = pd.concat([df_DROP_INVALID_AfterFS_ohe, df_DROP_INVALID_AfterFS_encoded], axis=1)

df_DROP_INVALID_AfterFS_ohe.head()

,UMUR (BULAN),banana_price,papaya_price,rice_price,bread_price,fish_price,chicken_price,carrot_price,tomato_price,cauliflower_price,milk_price,STATUS PEMAKANAN_Malpemakanan,STATUS PEMAKANAN_Normal,"JENIS TASKA_TASKA Agensi Kerajaan (GENIUS, KEMAS, JPNIN, YPKT)",JENIS TASKA_TASKA Di Rumah,JENIS TASKA_TASKA Di Tempat Kerja (Sektor Awam),JENIS TASKA_TASKA Di Tempat Kerja (Sektor Swasta),JENIS TASKA_TASKA Institusi,TASKA_LOKASI_BANDAR,TASKA_LOKASI_LUAR BANDAR
0,60,6.0,4.29,25.99,2.8,9.99,8.9,4.49,5.00,7.9,20.9,1.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0
1,53,6.0,4.29,25.99,2.8,9.99,8.9,4.49,5.00,7.9,20.9,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0
2,52,6.0,4.29,25.99,2.8,9.99,8.9,4.49,5.00,7.9,20.9,1.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0
3,58,6.0,4.50,25.90,2.8,10.00,8.9,5.00,5.49,8.0,20.9,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0
4,56,6.0,4.50,25.90,2.8,10.00,8.9,5.00,5.49,8.0,20.9,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0


In [29]:
x_ohe = df_DROP_INVALID_AfterFS_ohe
y_ohe = df_DROP_INVALID_AfterFS['BMI']

x_train_ohe, x_test_ohe, y_train_ohe, y_test_ohe = train_test_split(x_ohe, y_ohe, test_size = 0.30, random_state=42, stratify=y_ohe)

In [30]:
svm_model = svm.SVC(kernel='rbf', class_weight='balanced', random_state=42)
svm_model.fit(x_train_ohe, y_train_ohe)
y_pred = svm_model.predict(x_test_ohe)

In [31]:
# Evaluate the model
print("Confusion Matrix:")
print(confusion_matrix(y_test_ohe, y_pred))
print("\nClassification Report:")
print(classification_report(y_test_ohe, y_pred))

Confusion Matrix:
[[ 0 12  0 22 21  0]
 [ 0  4  0 13  7  0]
 [ 0  3  0  7 10  0]
 [ 0 13  0 16 11  0]
 [ 0  5  0  6  7  0]
 [ 0  2  0  4  4  0]]

Classification Report:
                               precision    recall  f1-score   support

           Berat badan normal       0.00      0.00      0.00        55
       Berlebihan berat badan       0.10      0.17      0.13        24
                         Obes       0.00      0.00      0.00        20
Risiko berlebihan berat badan       0.24      0.40      0.30        40
                        Susut       0.12      0.39      0.18        18
                  Susut teruk       0.00      0.00      0.00        10

                     accuracy                           0.16       167
                    macro avg       0.08      0.16      0.10       167
                 weighted avg       0.08      0.16      0.11       167



c:\Users\hp\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\hp\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\hp\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f

### Use SMOTE(Synthetic Minority Over-sampling Technique) to solve class imbalance

#### Label Encoder

In [32]:
sm = SMOTE(random_state=42)
X_train_res, y_train_res = sm.fit_resample(x_train_le, y_train_le)

In [33]:
svm_model = svm.SVC(kernel='rbf', class_weight='balanced', random_state=42)
svm_model.fit(X_train_res, y_train_res)
y_pred = svm_model.predict(x_test_le)

In [34]:
# Evaluate the model
print("Confusion Matrix:")
print(confusion_matrix(y_test_le, y_pred))
print("\nClassification Report:")
print(classification_report(y_test_le, y_pred))

Confusion Matrix:
[[ 0  6  0 31 14  4]
 [ 0  3  0 14  4  3]
 [ 0  0  0  8  4  8]
 [ 0  3  0 28  7  2]
 [ 0  3  0  8  6  1]
 [ 0  0  0  6  2  2]]

Classification Report:
              precision    recall  f1-score   support

           0       0.00      0.00      0.00        55
           1       0.20      0.12      0.15        24
           2       0.00      0.00      0.00        20
           3       0.29      0.70      0.41        40
           4       0.16      0.33      0.22        18
           5       0.10      0.20      0.13        10

    accuracy                           0.23       167
   macro avg       0.13      0.23      0.15       167
weighted avg       0.12      0.23      0.15       167



c:\Users\hp\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\hp\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\hp\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f

#### One-Hot Encoder

In [35]:
sm = SMOTE(random_state=42)
X_train_res, y_train_res = sm.fit_resample(x_train_ohe, y_train_ohe)

In [36]:
svm_model = svm.SVC(kernel='rbf', class_weight='balanced', random_state=42)
svm_model.fit(X_train_res, y_train_res)
y_pred = svm_model.predict(x_test_ohe)

In [37]:
# Evaluate the model
print("Confusion Matrix:")
print(confusion_matrix(y_test_ohe, y_pred))
print("\nClassification Report:")
print(classification_report(y_test_ohe, y_pred))

Confusion Matrix:
[[ 1  7  0 25 10 12]
 [ 0  3  0 13  5  3]
 [ 0  0  0  8  4  8]
 [ 0  3  0 24  6  7]
 [ 0  3  0  7  5  3]
 [ 0  1  0  5  1  3]]

Classification Report:
                               precision    recall  f1-score   support

           Berat badan normal       1.00      0.02      0.04        55
       Berlebihan berat badan       0.18      0.12      0.15        24
                         Obes       0.00      0.00      0.00        20
Risiko berlebihan berat badan       0.29      0.60      0.39        40
                        Susut       0.16      0.28      0.20        18
                  Susut teruk       0.08      0.30      0.13        10

                     accuracy                           0.22       167
                    macro avg       0.29      0.22      0.15       167
                 weighted avg       0.45      0.22      0.16       167



c:\Users\hp\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\hp\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\hp\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f

## Strategy: Replace Invalid Data with Mode

In [38]:
file_path = "Dataset/RP_MODE_AfterFS.csv"

# Read the first line to count the number of columns
with open(file_path, 'r', encoding='utf-8') as x:
    ncols = len(x.readline().strip().split(','))

# Load CSV using the correct number of columns
df_RP_MODE_AfterFS = pd.read_csv(file_path, usecols=range(0, ncols))

# Display the DataFrame
df_RP_MODE_AfterFS.head()

,UMUR (BULAN),STATUS PEMAKANAN,JENIS TASKA,TASKA_LOKASI,BMI,banana_price,papaya_price,rice_price,bread_price,fish_price,chicken_price,carrot_price,tomato_price,cauliflower_price,milk_price
0,60,Malpemakanan,TASKA Institusi,BANDAR,Risiko berlebihan berat badan,6.0,4.29,25.99,2.8,9.99,8.9,4.49,5.00,7.9,20.9
1,54,Malpemakanan,TASKA Institusi,BANDAR,Berlebihan berat badan,6.0,4.29,25.99,2.8,9.99,8.9,4.49,5.00,7.9,20.9
2,53,Malpemakanan,TASKA Di Rumah,BANDAR,Risiko berlebihan berat badan,6.0,4.29,25.99,2.8,9.99,8.9,4.49,5.00,7.9,20.9
3,52,Malpemakanan,TASKA Institusi,BANDAR,Obes,6.0,4.29,25.99,2.8,9.99,8.9,4.49,5.00,7.9,20.9
4,58,Malpemakanan,"TASKA Agensi Kerajaan (GENIUS, KEMAS, JPNIN, Y...",LUAR BANDAR,Berlebihan berat badan,6.0,4.50,25.90,2.8,10.00,8.9,5.00,5.49,8.0,20.9


### Encoding Strategy

#### Label Encoder

In [39]:
# Create a LabelEncoder object
le = LabelEncoder()
df_RP_MODE_AfterFS_le = df_RP_MODE_AfterFS.copy()

# Iterate over the columns of the DataFrame
for col in df_RP_MODE_AfterFS_le.columns:
    # Check if the column is of object type (categorical)
    if df_RP_MODE_AfterFS_le[col].dtype == 'object':
        # Fit and transform the column using LabelEncoder
        df_RP_MODE_AfterFS_le[col] = le.fit_transform(df_RP_MODE_AfterFS_le[col])

df_RP_MODE_AfterFS_le.head()

,UMUR (BULAN),STATUS PEMAKANAN,JENIS TASKA,TASKA_LOKASI,BMI,banana_price,papaya_price,rice_price,bread_price,fish_price,chicken_price,carrot_price,tomato_price,cauliflower_price,milk_price
0,60,0,4,0,3,6.0,4.29,25.99,2.8,9.99,8.9,4.49,5.00,7.9,20.9
1,54,0,4,0,1,6.0,4.29,25.99,2.8,9.99,8.9,4.49,5.00,7.9,20.9
2,53,0,1,0,3,6.0,4.29,25.99,2.8,9.99,8.9,4.49,5.00,7.9,20.9
3,52,0,4,0,2,6.0,4.29,25.99,2.8,9.99,8.9,4.49,5.00,7.9,20.9
4,58,0,0,1,1,6.0,4.50,25.90,2.8,10.00,8.9,5.00,5.49,8.0,20.9


In [40]:
x_le = df_RP_MODE_AfterFS_le.drop('BMI', axis=1)
y_le = df_RP_MODE_AfterFS_le['BMI']

x_train_le, x_test_le, y_train_le, y_test_le = train_test_split(x_le, y_le, test_size = 0.30, random_state=42, stratify=y_le)

In [41]:
svm_model = svm.SVC(kernel='rbf', class_weight='balanced', random_state=42)
svm_model.fit(x_train_le, y_train_le)
y_pred = svm_model.predict(x_test_le)

In [42]:
# Evaluate the model
print("Confusion Matrix:")
print(confusion_matrix(y_test_le, y_pred))
print("\nClassification Report:")
print(classification_report(y_test_le, y_pred))

Confusion Matrix:
[[44  0  0  0 21  0]
 [23  0  0  0  8  0]
 [14  0  0  0 10  3]
 [32  0  0  0 15  0]
 [14  0  0  0 10  0]
 [ 6  0  0  0  5  1]]

Classification Report:
              precision    recall  f1-score   support

           0       0.33      0.68      0.44        65
           1       0.00      0.00      0.00        31
           2       0.00      0.00      0.00        27
           3       0.00      0.00      0.00        47
           4       0.14      0.42      0.22        24
           5       0.25      0.08      0.12        12

    accuracy                           0.27       206
   macro avg       0.12      0.20      0.13       206
weighted avg       0.14      0.27      0.17       206



c:\Users\hp\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\hp\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\hp\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f

#### One-Hot Encoder

In [43]:
df_RP_MODE_AfterFS_ohe = df_RP_MODE_AfterFS.copy()
df_RP_MODE_AfterFS_ohe = df_RP_MODE_AfterFS_ohe.drop('BMI', axis=1)

In [44]:
# Create a OneHotEncoder object
ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=False)

# Fit and transform the categorical columns
categorical_cols = df_RP_MODE_AfterFS_ohe.select_dtypes(include=['object']).columns
df_RP_MODE_AfterFS_encoded = pd.DataFrame(ohe.fit_transform(df_RP_MODE_AfterFS_ohe[categorical_cols]))

# Get feature names for the encoded columns
encoded_feature_names = list(ohe.get_feature_names_out(categorical_cols))
df_RP_MODE_AfterFS_encoded.columns = encoded_feature_names

# Drop original categorical columns from the dataframe
df_RP_MODE_AfterFS_ohe = df_RP_MODE_AfterFS_ohe.drop(categorical_cols, axis=1)

# Concatenate the encoded columns with the remaining numerical features
df_RP_MODE_AfterFS_ohe = pd.concat([df_RP_MODE_AfterFS_ohe, df_RP_MODE_AfterFS_encoded], axis=1)

df_RP_MODE_AfterFS_ohe.head()

,UMUR (BULAN),banana_price,papaya_price,rice_price,bread_price,fish_price,chicken_price,carrot_price,tomato_price,cauliflower_price,milk_price,STATUS PEMAKANAN_Malpemakanan,STATUS PEMAKANAN_Normal,"JENIS TASKA_TASKA Agensi Kerajaan (GENIUS, KEMAS, JPNIN, YPKT)",JENIS TASKA_TASKA Di Rumah,JENIS TASKA_TASKA Di Tempat Kerja (Sektor Awam),JENIS TASKA_TASKA Di Tempat Kerja (Sektor Swasta),JENIS TASKA_TASKA Institusi,TASKA_LOKASI_BANDAR,TASKA_LOKASI_LUAR BANDAR
0,60,6.0,4.29,25.99,2.8,9.99,8.9,4.49,5.00,7.9,20.9,1.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0
1,54,6.0,4.29,25.99,2.8,9.99,8.9,4.49,5.00,7.9,20.9,1.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0
2,53,6.0,4.29,25.99,2.8,9.99,8.9,4.49,5.00,7.9,20.9,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0
3,52,6.0,4.29,25.99,2.8,9.99,8.9,4.49,5.00,7.9,20.9,1.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0
4,58,6.0,4.50,25.90,2.8,10.00,8.9,5.00,5.49,8.0,20.9,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0


In [45]:
x_ohe = df_RP_MODE_AfterFS_ohe
y_ohe = df_RP_MODE_AfterFS['BMI']

x_train_ohe, x_test_ohe, y_train_ohe, y_test_ohe = train_test_split( x_ohe, y_ohe, test_size = 0.30, random_state=42, stratify=y_ohe)

In [46]:
svm_model = svm.SVC(kernel='rbf', class_weight='balanced', random_state=42)
svm_model.fit(x_train_ohe, y_train_ohe)
y_pred = svm_model.predict(x_test_ohe)

In [47]:
# Evaluate the model
print("Confusion Matrix:")
print(confusion_matrix(y_test_ohe, y_pred))
print("\nClassification Report:")
print(classification_report(y_test_ohe, y_pred))

Confusion Matrix:
[[39  0  0  0 26  0]
 [21  0  0  0  8  2]
 [15  0  0  0 10  2]
 [31  0  0  0 15  1]
 [14  0  0  0 10  0]
 [ 6  0  0  0  5  1]]

Classification Report:
                               precision    recall  f1-score   support

           Berat badan normal       0.31      0.60      0.41        65
       Berlebihan berat badan       0.00      0.00      0.00        31
                         Obes       0.00      0.00      0.00        27
Risiko berlebihan berat badan       0.00      0.00      0.00        47
                        Susut       0.14      0.42      0.20        24
                  Susut teruk       0.17      0.08      0.11        12

                     accuracy                           0.24       206
                    macro avg       0.10      0.18      0.12       206
                 weighted avg       0.12      0.24      0.16       206



c:\Users\hp\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\hp\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\hp\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f

### Use SMOTE(Synthetic Minority Over-sampling Technique) to solve class imbalance

#### Label Encoder

In [48]:
sm = SMOTE(random_state=42)
X_train_res, y_train_res = sm.fit_resample(x_train_le, y_train_le)

In [49]:
svm_model = svm.SVC(kernel='rbf', class_weight='balanced', random_state=42)
svm_model.fit(X_train_res, y_train_res)
y_pred = svm_model.predict(x_test_le)

In [50]:
# Evaluate the model
print("Confusion Matrix:")
print(confusion_matrix(y_test_le, y_pred))
print("\nClassification Report:")
print(classification_report(y_test_le, y_pred))

Confusion Matrix:
[[ 0  9 15 10  3 28]
 [ 0  4  7  5  0 15]
 [ 0  2  9  1  0 15]
 [ 0  7  9  9  1 21]
 [ 0  1 10  8  0  5]
 [ 0  0  4  3  1  4]]

Classification Report:
              precision    recall  f1-score   support

           0       0.00      0.00      0.00        65
           1       0.17      0.13      0.15        31
           2       0.17      0.33      0.22        27
           3       0.25      0.19      0.22        47
           4       0.00      0.00      0.00        24
           5       0.05      0.33      0.08        12

    accuracy                           0.13       206
   macro avg       0.11      0.16      0.11       206
weighted avg       0.11      0.13      0.11       206



c:\Users\hp\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\hp\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\hp\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f

#### One-Hot Encoder

In [51]:
sm = SMOTE(random_state=42)
X_train_res, y_train_res = sm.fit_resample(x_train_ohe, y_train_ohe)

In [52]:
svm_model = svm.SVC(kernel='rbf', class_weight='balanced', random_state=42)
svm_model.fit(X_train_res, y_train_res)
y_pred = svm_model.predict(x_test_ohe)

In [53]:
# Evaluate the model
print("Confusion Matrix:")
print(confusion_matrix(y_test_ohe, y_pred))
print("\nClassification Report:")
print(classification_report(y_test_ohe, y_pred))

Confusion Matrix:
[[ 1 10 19  8  0 27]
 [ 0  5  7  1  0 18]
 [ 0  4 10  4  0  9]
 [ 0  4 13  8  0 22]
 [ 0  2 10  4  0  8]
 [ 0  2  5  2  0  3]]

Classification Report:
                               precision    recall  f1-score   support

           Berat badan normal       1.00      0.02      0.03        65
       Berlebihan berat badan       0.19      0.16      0.17        31
                         Obes       0.16      0.37      0.22        27
Risiko berlebihan berat badan       0.30      0.17      0.22        47
                        Susut       0.00      0.00      0.00        24
                  Susut teruk       0.03      0.25      0.06        12

                     accuracy                           0.13       206
                    macro avg       0.28      0.16      0.12       206
                 weighted avg       0.43      0.13      0.12       206



c:\Users\hp\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\hp\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\hp\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f

# Model Tuning

In [54]:
file_path = "Dataset/RP_MODE_AfterFS.csv"

# Read the first line to count the number of columns
with open(file_path, 'r', encoding='utf-8') as x:
    ncols = len(x.readline().strip().split(','))

# Load CSV using the correct number of columns
df_RP_MODE_AfterFS = pd.read_csv(file_path, usecols=range(0, ncols))

# Display the DataFrame
df_RP_MODE_AfterFS.head()

,UMUR (BULAN),STATUS PEMAKANAN,JENIS TASKA,TASKA_LOKASI,BMI,banana_price,papaya_price,rice_price,bread_price,fish_price,chicken_price,carrot_price,tomato_price,cauliflower_price,milk_price
0,60,Malpemakanan,TASKA Institusi,BANDAR,Risiko berlebihan berat badan,6.0,4.29,25.99,2.8,9.99,8.9,4.49,5.00,7.9,20.9
1,54,Malpemakanan,TASKA Institusi,BANDAR,Berlebihan berat badan,6.0,4.29,25.99,2.8,9.99,8.9,4.49,5.00,7.9,20.9
2,53,Malpemakanan,TASKA Di Rumah,BANDAR,Risiko berlebihan berat badan,6.0,4.29,25.99,2.8,9.99,8.9,4.49,5.00,7.9,20.9
3,52,Malpemakanan,TASKA Institusi,BANDAR,Obes,6.0,4.29,25.99,2.8,9.99,8.9,4.49,5.00,7.9,20.9
4,58,Malpemakanan,"TASKA Agensi Kerajaan (GENIUS, KEMAS, JPNIN, Y...",LUAR BANDAR,Berlebihan berat badan,6.0,4.50,25.90,2.8,10.00,8.9,5.00,5.49,8.0,20.9


In [55]:
# Create a LabelEncoder object
le = LabelEncoder()
df_RP_MODE_AfterFS_le = df_RP_MODE_AfterFS.copy()

# Iterate over the columns of the DataFrame
for col in df_RP_MODE_AfterFS_le.columns:
    # Check if the column is of object type (categorical)
    if df_RP_MODE_AfterFS_le[col].dtype == 'object':
        # Fit and transform the column using LabelEncoder
        df_RP_MODE_AfterFS_le[col] = le.fit_transform(df_RP_MODE_AfterFS_le[col])

df_RP_MODE_AfterFS_le.head()

,UMUR (BULAN),STATUS PEMAKANAN,JENIS TASKA,TASKA_LOKASI,BMI,banana_price,papaya_price,rice_price,bread_price,fish_price,chicken_price,carrot_price,tomato_price,cauliflower_price,milk_price
0,60,0,4,0,3,6.0,4.29,25.99,2.8,9.99,8.9,4.49,5.00,7.9,20.9
1,54,0,4,0,1,6.0,4.29,25.99,2.8,9.99,8.9,4.49,5.00,7.9,20.9
2,53,0,1,0,3,6.0,4.29,25.99,2.8,9.99,8.9,4.49,5.00,7.9,20.9
3,52,0,4,0,2,6.0,4.29,25.99,2.8,9.99,8.9,4.49,5.00,7.9,20.9
4,58,0,0,1,1,6.0,4.50,25.90,2.8,10.00,8.9,5.00,5.49,8.0,20.9


In [56]:
x_le = df_RP_MODE_AfterFS_le.drop('BMI', axis=1)
y_le = df_RP_MODE_AfterFS_le['BMI']

x_train, x_test, y_train, y_test = train_test_split(x_le, y_le, test_size = 0.30, random_state=42, stratify=y_le)

### First Round

In [57]:
param_grid = [
    # Linear kernel: just C
    {
        'kernel': ['linear'],
        'C': [0.1, 1, 10]
    },

    # RBF kernel: C and gamma
    {
        'kernel': ['rbf'],
        'C': [1, 10],
        'gamma': ['scale', 0.1]
    },

    # Polynomial kernel: small grid for degree and coef0
    {
        'kernel': ['poly'],
        'C': [1, 10],
        'gamma': ['scale'],
        'degree': [2, 3],
        'coef0': [0.0]
    },

    # Sigmoid kernel: minimal gamma and coef0 search
    {
        'kernel': ['sigmoid'],
        'C': [1, 10],
        'gamma': ['scale'],
        'coef0': [0.0]
    }
]

In [58]:
svc = svm.SVC()
grid_search = GridSearchCV(svc,
                           param_grid=param_grid,
                           scoring='accuracy',
                           cv=5,
                           n_jobs=-1,
                           verbose=2
)

grid_search.fit(x_train, y_train)

Fitting 5 folds for each of 13 candidates, totalling 65 fits


GridSearchCV(cv=5, estimator=SVC(), n_jobs=-1,
             param_grid=[{'C': [0.1, 1, 10], 'kernel': ['linear']},
                         {'C': [1, 10], 'gamma': ['scale', 0.1],
                          'kernel': ['rbf']},
                         {'C': [1, 10], 'coef0': [0.0], 'degree': [2, 3],
                          'gamma': ['scale'], 'kernel': ['poly']},
                         {'C': [1, 10], 'coef0': [0.0], 'gamma': ['scale'],
                          'kernel': ['sigmoid']}],
             scoring='accuracy', verbose=2)

In [59]:
# Print the best parameters and the corresponding best score
print("Best parameters found: ", grid_search.best_params_)
print("Best accuracy found: ", grid_search.best_score_)
print("Best kernel: ", grid_search.best_estimator_.kernel)

Best parameters found:  {'C': 0.1, 'kernel': 'linear'}
Best accuracy found:  0.37864035087719305
Best kernel:  linear


In [60]:
# Use the best estimator to predict on the test set
best_svm_model = grid_search.best_estimator_
y_pred_tuned = best_svm_model.predict(x_test)

# Evaluate the tuned model
print("\nConfusion Matrix (Tuned Model):")
print(confusion_matrix(y_test, y_pred_tuned))
print("\nClassification Report (Tuned Model):")
print(classification_report(y_test, y_pred_tuned))


Confusion Matrix (Tuned Model):
[[34  0  0 31  0  0]
 [ 0  0  0 31  0  0]
 [ 0  0  0 27  0  0]
 [ 4  0  0 43  0  0]
 [ 3  0  0 21  0  0]
 [ 1  0  0 11  0  0]]

Classification Report (Tuned Model):
              precision    recall  f1-score   support

           0       0.81      0.52      0.64        65
           1       0.00      0.00      0.00        31
           2       0.00      0.00      0.00        27
           3       0.26      0.91      0.41        47
           4       0.00      0.00      0.00        24
           5       0.00      0.00      0.00        12

    accuracy                           0.37       206
   macro avg       0.18      0.24      0.17       206
weighted avg       0.32      0.37      0.29       206



c:\Users\hp\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\hp\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\hp\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f

### Second Round

In [61]:
param_grid = {
    'kernel': ['linear'],
    'C': [0.001, 0.01, 0.1, 1, 10, 100],  # Regularization strength
    'class_weight': [None, 'balanced'],  # Handle imbalance
    'tol': [1e-4, 1e-3, 1e-2],            # Convergence tolerance
    'max_iter': [1000, 5000, 10000],      # Max training iterations
    'shrinking': [True, False],          # Whether to use shrinking heuristic
}

In [62]:
svc = svm.SVC()
grid_search = GridSearchCV(svc,
                           param_grid=param_grid,
                           scoring='accuracy',
                           cv=5,
                           n_jobs=-1,
                           verbose=2
)

grid_search.fit(x_train, y_train)

Fitting 5 folds for each of 216 candidates, totalling 1080 fits


c:\Users\hp\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\svm\_base.py:305: ConvergenceWarning: Solver terminated early (max_iter=5000).  Consider pre-processing your data with StandardScaler or MinMaxScaler.
  warnings.warn(


GridSearchCV(cv=5, estimator=SVC(), n_jobs=-1,
             param_grid={'C': [0.001, 0.01, 0.1, 1, 10, 100],
                         'class_weight': [None, 'balanced'],
                         'kernel': ['linear'], 'max_iter': [1000, 5000, 10000],
                         'shrinking': [True, False],
                         'tol': [0.0001, 0.001, 0.01]},
             scoring='accuracy', verbose=2)

In [63]:
# Print the best parameters and the corresponding best score
print("Best parameters found: ", grid_search.best_params_)
print("Best accuracy found: ", grid_search.best_score_)

Best parameters found:  {'C': 1, 'class_weight': None, 'kernel': 'linear', 'max_iter': 5000, 'shrinking': True, 'tol': 0.001}
Best accuracy found:  0.384890350877193


In [64]:
# Use the best estimator to predict on the test set
best_svm_model = grid_search.best_estimator_
y_pred_tuned = best_svm_model.predict(x_test)

# Evaluate the tuned model
print("\nConfusion Matrix (Tuned Model):")
print(confusion_matrix(y_test, y_pred_tuned))
print("\nClassification Report (Tuned Model):")
print(classification_report(y_test, y_pred_tuned))


Confusion Matrix (Tuned Model):
[[34  0  7 24  0  0]
 [ 0  0  8 23  0  0]
 [ 0  0 11 16  0  0]
 [ 4  0  6 37  0  0]
 [ 3  3  5 13  0  0]
 [ 1  0  6  5  0  0]]

Classification Report (Tuned Model):
              precision    recall  f1-score   support

           0       0.81      0.52      0.64        65
           1       0.00      0.00      0.00        31
           2       0.26      0.41      0.31        27
           3       0.31      0.79      0.45        47
           4       0.00      0.00      0.00        24
           5       0.00      0.00      0.00        12

    accuracy                           0.40       206
   macro avg       0.23      0.29      0.23       206
weighted avg       0.36      0.40      0.34       206



c:\Users\hp\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\hp\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\hp\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f

# Number of Target Labels

In [65]:
file_path = "Dataset/RP_MODE_AfterFS.csv"

# Read the first line to count the number of columns
with open(file_path, 'r', encoding='utf-8') as x:
    ncols = len(x.readline().strip().split(','))

# Load CSV using the correct number of columns
df_RP_MODE_AfterFS = pd.read_csv(file_path, usecols=range(0, ncols))

# Display the DataFrame
df_RP_MODE_AfterFS.head()

,UMUR (BULAN),STATUS PEMAKANAN,JENIS TASKA,TASKA_LOKASI,BMI,banana_price,papaya_price,rice_price,bread_price,fish_price,chicken_price,carrot_price,tomato_price,cauliflower_price,milk_price
0,60,Malpemakanan,TASKA Institusi,BANDAR,Risiko berlebihan berat badan,6.0,4.29,25.99,2.8,9.99,8.9,4.49,5.00,7.9,20.9
1,54,Malpemakanan,TASKA Institusi,BANDAR,Berlebihan berat badan,6.0,4.29,25.99,2.8,9.99,8.9,4.49,5.00,7.9,20.9
2,53,Malpemakanan,TASKA Di Rumah,BANDAR,Risiko berlebihan berat badan,6.0,4.29,25.99,2.8,9.99,8.9,4.49,5.00,7.9,20.9
3,52,Malpemakanan,TASKA Institusi,BANDAR,Obes,6.0,4.29,25.99,2.8,9.99,8.9,4.49,5.00,7.9,20.9
4,58,Malpemakanan,"TASKA Agensi Kerajaan (GENIUS, KEMAS, JPNIN, Y...",LUAR BANDAR,Berlebihan berat badan,6.0,4.50,25.90,2.8,10.00,8.9,5.00,5.49,8.0,20.9


## 4 Labels

In [117]:
df = df_RP_MODE_AfterFS.replace({'BMI': {"Berat badan normal": "Normal",
                                                 "Berlebihan berat badan": "Berat berlebihan",
                                                 "Obes": "Berat berlebihan",
                                                 "Risiko berlebihan berat badan": "Berisiko",
                                                 "Susut": "Kurang berat badan",
                                                 "Susut teruk": "Kurang berat badan"}})

In [118]:
# Create a LabelEncoder object
le = LabelEncoder()
df_RP_MODE_AfterFS_le = df.copy()

# Iterate over the columns of the DataFrame
for col in df_RP_MODE_AfterFS_le.columns:
    # Check if the column is of object type (categorical)
    if df_RP_MODE_AfterFS_le[col].dtype == 'object':
        # Fit and transform the column using LabelEncoder
        df_RP_MODE_AfterFS_le[col] = le.fit_transform(df_RP_MODE_AfterFS_le[col])

df_RP_MODE_AfterFS_le.head()

,UMUR (BULAN),STATUS PEMAKANAN,JENIS TASKA,TASKA_LOKASI,BMI,banana_price,papaya_price,rice_price,bread_price,fish_price,chicken_price,carrot_price,tomato_price,cauliflower_price,milk_price
0,60,0,4,0,1,6.0,4.29,25.99,2.8,9.99,8.9,4.49,5.00,7.9,20.9
1,54,0,4,0,0,6.0,4.29,25.99,2.8,9.99,8.9,4.49,5.00,7.9,20.9
2,53,0,1,0,1,6.0,4.29,25.99,2.8,9.99,8.9,4.49,5.00,7.9,20.9
3,52,0,4,0,0,6.0,4.29,25.99,2.8,9.99,8.9,4.49,5.00,7.9,20.9
4,58,0,0,1,0,6.0,4.50,25.90,2.8,10.00,8.9,5.00,5.49,8.0,20.9


In [119]:
x_le = df_RP_MODE_AfterFS_le.drop('BMI', axis=1)
y_le = df['BMI']

x_train, x_test, y_train, y_test = train_test_split(x_le, y_le, test_size = 0.30, random_state=42, stratify=y_le)

In [120]:
svm_model = svm.SVC(kernel='linear', class_weight=None, C= 1, max_iter=5000, shrinking=True, tol=0.0001, random_state=42)
svm_model.fit(x_train, y_train)
y_pred = svm_model.predict(x_test)

c:\Users\hp\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\svm\_base.py:305: ConvergenceWarning: Solver terminated early (max_iter=5000).  Consider pre-processing your data with StandardScaler or MinMaxScaler.
  warnings.warn(


In [121]:
# Evaluate the model
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Confusion Matrix:
[[38  8  7  5]
 [19 15  6  7]
 [21  2  7  6]
 [14 12  2 37]]

Classification Report:
                    precision    recall  f1-score   support

  Berat berlebihan       0.41      0.66      0.51        58
          Berisiko       0.41      0.32      0.36        47
Kurang berat badan       0.32      0.19      0.24        36
            Normal       0.67      0.57      0.62        65

          accuracy                           0.47       206
         macro avg       0.45      0.43      0.43       206
      weighted avg       0.48      0.47      0.46       206



## 3 Labels

In [122]:
df = df_RP_MODE_AfterFS.replace({'BMI': {"Berat badan normal": "Normal",
                                                 "Berlebihan berat badan": "Berat berlebihan",
                                                 "Obes": "Berat berlebihan",
                                                 "Risiko berlebihan berat badan": "Berat berlebihan",
                                                 "Susut": "Kurang berat badan",
                                                 "Susut teruk": "Kurang berat badan"}})

In [123]:
# Create a LabelEncoder object
le = LabelEncoder()
df_RP_MODE_AfterFS_le = df.copy()

# Iterate over the columns of the DataFrame
for col in df_RP_MODE_AfterFS_le.columns:
    # Check if the column is of object type (categorical)
    if df_RP_MODE_AfterFS_le[col].dtype == 'object':
        # Fit and transform the column using LabelEncoder
        df_RP_MODE_AfterFS_le[col] = le.fit_transform(df_RP_MODE_AfterFS_le[col])

df_RP_MODE_AfterFS_le.head()

,UMUR (BULAN),STATUS PEMAKANAN,JENIS TASKA,TASKA_LOKASI,BMI,banana_price,papaya_price,rice_price,bread_price,fish_price,chicken_price,carrot_price,tomato_price,cauliflower_price,milk_price
0,60,0,4,0,0,6.0,4.29,25.99,2.8,9.99,8.9,4.49,5.00,7.9,20.9
1,54,0,4,0,0,6.0,4.29,25.99,2.8,9.99,8.9,4.49,5.00,7.9,20.9
2,53,0,1,0,0,6.0,4.29,25.99,2.8,9.99,8.9,4.49,5.00,7.9,20.9
3,52,0,4,0,0,6.0,4.29,25.99,2.8,9.99,8.9,4.49,5.00,7.9,20.9
4,58,0,0,1,0,6.0,4.50,25.90,2.8,10.00,8.9,5.00,5.49,8.0,20.9


In [124]:
x_le = df_RP_MODE_AfterFS_le.drop('BMI', axis=1)
y_le = df['BMI']

x_train, x_test, y_train, y_test = train_test_split(x_le, y_le, test_size = 0.30, random_state=42, stratify=y_le)

In [125]:
svm_model = svm.SVC(kernel='linear', class_weight=None, C= 1, max_iter=5000, shrinking=True, tol=0.0001, random_state=42)
svm_model.fit(x_train, y_train)
y_pred = svm_model.predict(x_test)

c:\Users\hp\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\svm\_base.py:305: ConvergenceWarning: Solver terminated early (max_iter=5000).  Consider pre-processing your data with StandardScaler or MinMaxScaler.
  warnings.warn(


In [126]:
# Evaluate the model
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Confusion Matrix:
[[88  7 10]
 [26  3  7]
 [23  1 41]]

Classification Report:
                    precision    recall  f1-score   support

  Berat berlebihan       0.64      0.84      0.73       105
Kurang berat badan       0.27      0.08      0.13        36
            Normal       0.71      0.63      0.67        65

          accuracy                           0.64       206
         macro avg       0.54      0.52      0.51       206
      weighted avg       0.60      0.64      0.60       206



In [127]:
df.to_csv('Dataset/SVM_model_3labels.csv', index=False)

## 2 Labels

In [128]:
df = df_RP_MODE_AfterFS.replace({'BMI': {"Berat badan normal": "Normal",
                                                 "Berlebihan berat badan": "Abnormal",
                                                 "Obes": "Abnormal",
                                                 "Risiko berlebihan berat badan": "Abnormal",
                                                 "Susut": "Abnormal",
                                                 "Susut teruk": "Abnormal"}})

In [129]:
# Create a LabelEncoder object
le = LabelEncoder()
df_RP_MODE_AfterFS_le = df.copy()

# Iterate over the columns of the DataFrame
for col in df_RP_MODE_AfterFS_le.columns:
    # Check if the column is of object type (categorical)
    if df_RP_MODE_AfterFS_le[col].dtype == 'object':
        # Fit and transform the column using LabelEncoder
        df_RP_MODE_AfterFS_le[col] = le.fit_transform(df_RP_MODE_AfterFS_le[col])

df_RP_MODE_AfterFS_le.head()

,UMUR (BULAN),STATUS PEMAKANAN,JENIS TASKA,TASKA_LOKASI,BMI,banana_price,papaya_price,rice_price,bread_price,fish_price,chicken_price,carrot_price,tomato_price,cauliflower_price,milk_price
0,60,0,4,0,0,6.0,4.29,25.99,2.8,9.99,8.9,4.49,5.00,7.9,20.9
1,54,0,4,0,0,6.0,4.29,25.99,2.8,9.99,8.9,4.49,5.00,7.9,20.9
2,53,0,1,0,0,6.0,4.29,25.99,2.8,9.99,8.9,4.49,5.00,7.9,20.9
3,52,0,4,0,0,6.0,4.29,25.99,2.8,9.99,8.9,4.49,5.00,7.9,20.9
4,58,0,0,1,0,6.0,4.50,25.90,2.8,10.00,8.9,5.00,5.49,8.0,20.9


In [130]:
x_le = df_RP_MODE_AfterFS_le.drop('BMI', axis=1)
y_le = df['BMI']

x_train, x_test, y_train, y_test = train_test_split(x_le, y_le, test_size = 0.30, random_state=42, stratify=y_le)

In [131]:
svm_model = svm.SVC(kernel='linear', class_weight=None, C= 1, max_iter=5000, shrinking=True, tol=0.0001, random_state=42)
svm_model.fit(x_train, y_train)
y_pred = svm_model.predict(x_test)

c:\Users\hp\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\svm\_base.py:305: ConvergenceWarning: Solver terminated early (max_iter=5000).  Consider pre-processing your data with StandardScaler or MinMaxScaler.
  warnings.warn(


In [132]:
# Evaluate the model
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Confusion Matrix:
[[125  16]
 [ 29  36]]

Classification Report:
              precision    recall  f1-score   support

    Abnormal       0.81      0.89      0.85       141
      Normal       0.69      0.55      0.62        65

    accuracy                           0.78       206
   macro avg       0.75      0.72      0.73       206
weighted avg       0.77      0.78      0.77       206



# Stratified K-Fold on 3 Labels SVM Model

In [133]:
file_path = "Dataset/svm_model_3labels.csv"

# Read the first line to count the number of columns
with open(file_path, 'r', encoding='utf-8') as x:
    ncols = len(x.readline().strip().split(','))

# Load CSV using the correct number of columns
df_svm_model_3labels = pd.read_csv(file_path, usecols=range(0, ncols))

# Display the DataFrame
df_svm_model_3labels.head()

,UMUR (BULAN),STATUS PEMAKANAN,JENIS TASKA,TASKA_LOKASI,BMI,banana_price,papaya_price,rice_price,bread_price,fish_price,chicken_price,carrot_price,tomato_price,cauliflower_price,milk_price
0,60,Malpemakanan,TASKA Institusi,BANDAR,Berat berlebihan,6.0,4.29,25.99,2.8,9.99,8.9,4.49,5.00,7.9,20.9
1,54,Malpemakanan,TASKA Institusi,BANDAR,Berat berlebihan,6.0,4.29,25.99,2.8,9.99,8.9,4.49,5.00,7.9,20.9
2,53,Malpemakanan,TASKA Di Rumah,BANDAR,Berat berlebihan,6.0,4.29,25.99,2.8,9.99,8.9,4.49,5.00,7.9,20.9
3,52,Malpemakanan,TASKA Institusi,BANDAR,Berat berlebihan,6.0,4.29,25.99,2.8,9.99,8.9,4.49,5.00,7.9,20.9
4,58,Malpemakanan,"TASKA Agensi Kerajaan (GENIUS, KEMAS, JPNIN, Y...",LUAR BANDAR,Berat berlebihan,6.0,4.50,25.90,2.8,10.00,8.9,5.00,5.49,8.0,20.9


In [134]:
df = df_svm_model_3labels.copy()
df = df.drop('BMI', axis=1)

In [135]:
# Create a LabelEncoder object
le = LabelEncoder()

# Iterate over the columns of the DataFrame
for col in df.columns:
    # Check if the column is of object type (categorical)
    if df[col].dtype == 'object':
        # Fit and transform the column using LabelEncoder
        df[col] = le.fit_transform(df[col])

df.head()

,UMUR (BULAN),STATUS PEMAKANAN,JENIS TASKA,TASKA_LOKASI,banana_price,papaya_price,rice_price,bread_price,fish_price,chicken_price,carrot_price,tomato_price,cauliflower_price,milk_price
0,60,0,4,0,6.0,4.29,25.99,2.8,9.99,8.9,4.49,5.00,7.9,20.9
1,54,0,4,0,6.0,4.29,25.99,2.8,9.99,8.9,4.49,5.00,7.9,20.9
2,53,0,1,0,6.0,4.29,25.99,2.8,9.99,8.9,4.49,5.00,7.9,20.9
3,52,0,4,0,6.0,4.29,25.99,2.8,9.99,8.9,4.49,5.00,7.9,20.9
4,58,0,0,1,6.0,4.50,25.90,2.8,10.00,8.9,5.00,5.49,8.0,20.9


In [136]:
x_le = df_RP_MODE_AfterFS_le.drop('BMI', axis=1)
y_le = df_svm_model_3labels['BMI']

x_train, x_test, y_train, y_test = train_test_split(x_le, y_le, test_size = 0.30, random_state=42, stratify=y_le)

In [137]:
svm_model = svm.SVC(kernel='linear', class_weight=None, C= 1, max_iter=5000, shrinking=True, tol=0.0001, random_state=42)
svm_model.fit(x_train, y_train)
y_pred = svm_model.predict(x_test)

c:\Users\hp\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\svm\_base.py:305: ConvergenceWarning: Solver terminated early (max_iter=5000).  Consider pre-processing your data with StandardScaler or MinMaxScaler.
  warnings.warn(


In [138]:
# Setup Stratified K-Fold
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

In [139]:
# Store fold-level metrics
report_sum = defaultdict(lambda: defaultdict(float))
accuracy_sum = 0.0
n_folds = skf.get_n_splits()

In [140]:
fold = 1

for train_idx, test_idx in skf.split(x_train, y_train):
    X_train_fold = x_train.iloc[train_idx]
    X_test_fold = x_train.iloc[test_idx]
    y_train_fold = y_train.iloc[train_idx]
    y_test_fold = y_train.iloc[test_idx]
    
    svm_model.fit(X_train_fold, y_train_fold)
    preds = svm_model.predict(X_test_fold)
    
    acc = accuracy_score(y_test_fold, preds)

    print(f"\nFold {fold} Accuracy: {acc:.4f}")
    report = classification_report(y_test_fold, preds, output_dict=True)
    print(f"Classification Report (Fold {fold}):")
    print(pd.DataFrame(report).T)
    
    # Sum reports
    for label, metrics in report.items():
        if label == 'accuracy':
            accuracy_sum += metrics
        else:
            for metric_name, value in metrics.items():
                report_sum[label][metric_name] += value
    fold += 1

c:\Users\hp\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\svm\_base.py:305: ConvergenceWarning: Solver terminated early (max_iter=5000).  Consider pre-processing your data with StandardScaler or MinMaxScaler.
  warnings.warn(
c:\Users\hp\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\svm\_base.py:305: ConvergenceWarning: Solver terminated early (max_iter=5000).  Consider pre-processing your data with StandardScaler or MinMaxScaler.
  warnings.warn(
c:\Users\hp\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\hp\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined a


Fold 1 Accuracy: 0.5833
Classification Report (Fold 1):
                    precision    recall  f1-score    support
Berat berlebihan     0.577465  0.854167  0.689076  48.000000
Kurang berat badan   0.200000  0.058824  0.090909  17.000000
Normal               0.700000  0.451613  0.549020  31.000000
accuracy             0.583333  0.583333  0.583333   0.583333
macro avg            0.492488  0.454868  0.443001  96.000000
weighted avg         0.550191  0.583333  0.537924  96.000000

Fold 2 Accuracy: 0.6354
Classification Report (Fold 2):
                    precision    recall  f1-score    support
Berat berlebihan     0.608108  0.918367  0.731707  49.000000
Kurang berat badan   0.000000  0.000000  0.000000  17.000000
Normal               0.727273  0.533333  0.615385  30.000000
accuracy             0.635417  0.635417  0.635417   0.635417
macro avg            0.445127  0.483900  0.449031  96.000000
weighted avg         0.537661  0.635417  0.565783  96.000000


c:\Users\hp\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\svm\_base.py:305: ConvergenceWarning: Solver terminated early (max_iter=5000).  Consider pre-processing your data with StandardScaler or MinMaxScaler.
  warnings.warn(
c:\Users\hp\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\hp\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\hp\AppData\Local\Programs\Python\Python313\Lib\site


Fold 3 Accuracy: 0.6458
Classification Report (Fold 3):
                    precision    recall  f1-score    support
Berat berlebihan     0.628571  0.897959  0.739496  49.000000
Kurang berat badan   0.000000  0.000000  0.000000  17.000000
Normal               0.692308  0.600000  0.642857  30.000000
accuracy             0.645833  0.645833  0.645833   0.645833
macro avg            0.440293  0.499320  0.460784  96.000000
weighted avg         0.537179  0.645833  0.578344  96.000000

Fold 4 Accuracy: 0.6947
Classification Report (Fold 4):
                    precision    recall  f1-score    support
Berat berlebihan     0.698413  0.897959  0.785714  49.000000
Kurang berat badan   0.428571  0.187500  0.260870  16.000000
Normal               0.760000  0.633333  0.690909  30.000000
accuracy             0.694737  0.694737  0.694737   0.694737
macro avg            0.628995  0.572931  0.579164  95.000000
weighted avg         0.672414  0.694737  0.667381  95.000000

Fold 5 Accuracy: 0.6316
Classif

c:\Users\hp\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\svm\_base.py:305: ConvergenceWarning: Solver terminated early (max_iter=5000).  Consider pre-processing your data with StandardScaler or MinMaxScaler.
  warnings.warn(


In [141]:
# Compute average classification report
print("\nAverage Classification Report Over All Folds:")
avg_report = {}
for label, metrics in report_sum.items():
    avg_report[label] = {}
    for metric_name, total_value in metrics.items():
        avg_report[label][metric_name] = total_value / n_folds

# Add average accuracy
avg_report['accuracy'] = {'score': accuracy_sum / n_folds}

print(pd.DataFrame(avg_report).T)


Average Classification Report Over All Folds:
                    precision    recall  f1-score  support    score
Berat berlebihan     0.628226  0.893282  0.737098     48.8      NaN
Kurang berat badan   0.192381  0.074265  0.106719     16.6      NaN
Normal               0.723285  0.536989  0.613920     30.2      NaN
macro avg            0.514630  0.501512  0.485912     95.6      NaN
weighted avg         0.582097  0.638180  0.588386     95.6      NaN
accuracy                  NaN       NaN       NaN      NaN  0.63818
